# 08. Submission Pipeline

This notebook builds and validates competition submissions from saved experiment predictions.

The first submission candidate uses:

- 50% CatBoost probabilities
- 50% XGBoost probabilities
- decision threshold: 0.130

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
EXPERIMENTS_DIR = PROJECT_ROOT / "artifacts" / "experiments"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"

COMPARISON_DIR = (
    EXPERIMENTS_DIR / "comparison_v0_raw_minimal"
)

BLEND_PREDICTIONS_PATH = (
    COMPARISON_DIR
    / "selected_blend_test_predictions.parquet"
)

SAMPLE_SUBMISSION_PATH = (
    DATA_DIR
    / "raw"
    / "final_proj_sample_submission.csv"
)

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Project root:          {PROJECT_ROOT}")
print(f"Comparison directory: {COMPARISON_DIR}")
print(f"Submissions directory:{SUBMISSIONS_DIR}")

Project root:          C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final
Comparison directory: C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\comparison_v0_raw_minimal
Submissions directory:C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\submissions


In [2]:
if not BLEND_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"Blend predictions not found: "
        f"{BLEND_PREDICTIONS_PATH}"
    )

if not SAMPLE_SUBMISSION_PATH.exists():
    raise FileNotFoundError(
        f"Sample submission not found: "
        f"{SAMPLE_SUBMISSION_PATH}"
    )

blend_predictions = pd.read_parquet(
    BLEND_PREDICTIONS_PATH
)

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH
)

print(
    "Blend predictions:",
    blend_predictions.shape,
)

print(
    "Sample submission:",
    sample_submission.shape,
)

print(
    "Sample submission columns:",
    sample_submission.columns.tolist(),
)

display(blend_predictions.head())
display(sample_submission.head())

Blend predictions: (2500, 3)
Sample submission: (2500, 2)
Sample submission columns: ['index', 'y']


,row_index,probability,prediction
0,0,0.001349,0
1,1,0.030158,0
2,2,0.005894,0
3,3,0.009900,0
4,4,0.004379,0


,index,y
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [3]:
required_prediction_columns = {
    "row_index",
    "probability",
    "prediction",
}

missing_prediction_columns = (
    required_prediction_columns
    - set(blend_predictions.columns)
)

if missing_prediction_columns:
    raise ValueError(
        "Missing prediction columns: "
        f"{sorted(missing_prediction_columns)}"
    )

if len(blend_predictions) != len(sample_submission):
    raise ValueError(
        "Prediction row count does not match "
        "sample submission row count: "
        f"{len(blend_predictions)} != "
        f"{len(sample_submission)}"
    )

if blend_predictions["row_index"].duplicated().any():
    raise ValueError(
        "Blend predictions contain duplicate row indices."
    )

expected_row_indices = pd.Series(
    range(len(blend_predictions)),
    name="row_index",
)

actual_row_indices = (
    blend_predictions["row_index"]
    .reset_index(drop=True)
)

if not actual_row_indices.equals(expected_row_indices):
    raise ValueError(
        "Blend prediction rows are not in the expected "
        "0..n-1 order."
    )

if not blend_predictions[
    "probability"
].between(0.0, 1.0).all():
    raise ValueError(
        "Probabilities must be between 0 and 1."
    )

if not set(
    blend_predictions["prediction"].unique()
).issubset({0, 1}):
    raise ValueError(
        "Predictions must contain only classes 0 and 1."
    )

print("Initial submission validation passed.")

Initial submission validation passed.


In [4]:
sample_indices = (
    sample_submission["index"]
    .reset_index(drop=True)
)

prediction_indices = (
    blend_predictions["row_index"]
    .reset_index(drop=True)
)

if not sample_indices.equals(prediction_indices):
    mismatch_mask = sample_indices != prediction_indices

    raise ValueError(
        "Sample submission indices do not match prediction rows. "
        f"First mismatches:\n"
        f"{pd.DataFrame({
            'sample_index': sample_indices[mismatch_mask],
            'prediction_index': prediction_indices[mismatch_mask],
        }).head()}"
    )

print("Sample submission and prediction indices are aligned.")

Sample submission and prediction indices are aligned.


## Build Submission

The official sample submission is used as the structural template. Only the target column is replaced with predictions from the selected CatBoost/XGBoost blend.

In [5]:
submission = sample_submission.copy()

submission["y"] = (
    blend_predictions["prediction"]
    .astype("int8")
    .to_numpy()
)

display(submission.head())

print(f"Submission shape:        {submission.shape}")
print(f"Predicted positive rows: {submission['y'].sum():,}")
print(f"Predicted positive rate: {submission['y'].mean():.2%}")

,index,y
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


Submission shape:        (2500, 2)
Predicted positive rows: 509
Predicted positive rate: 20.36%


In [6]:
expected_columns = sample_submission.columns.tolist()

if submission.columns.tolist() != expected_columns:
    raise ValueError(
        "Submission columns differ from the official template: "
        f"{submission.columns.tolist()} != {expected_columns}"
    )

if len(submission) != len(sample_submission):
    raise ValueError(
        "Submission row count differs from the sample submission."
    )

if submission["index"].isna().any():
    raise ValueError("Submission index contains missing values.")

if submission["index"].duplicated().any():
    raise ValueError("Submission index contains duplicate values.")

if submission["y"].isna().any():
    raise ValueError("Submission target contains missing values.")

if not set(submission["y"].unique()).issubset({0, 1}):
    raise ValueError(
        "Submission target must contain only 0 and 1."
    )

if not submission["index"].equals(
    sample_submission["index"]
):
    raise ValueError(
        "Submission index order differs from the official template."
    )

print("Final submission validation passed.")

Final submission validation passed.


In [7]:
SUBMISSION_NAME = (
    "catboost_xgboost_50_50_threshold_0130.csv"
)

submission_path = SUBMISSIONS_DIR / SUBMISSION_NAME

submission.to_csv(
    submission_path,
    index=False,
)

print(f"Submission saved to: {submission_path}")

Submission saved to: C:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\submissions\catboost_xgboost_50_50_threshold_0130.csv


In [8]:
saved_submission = pd.read_csv(submission_path)

if not saved_submission.equals(
    submission.astype(
        {
            "index": saved_submission["index"].dtype,
            "y": saved_submission["y"].dtype,
        }
    )
):
    raise ValueError(
        "Saved submission differs from the in-memory dataframe."
    )

print(f"Saved file size: {submission_path.stat().st_size:,} bytes")
print("Saved submission successfully verified.")

display(saved_submission.head())
display(saved_submission.tail())

Saved file size: 18,899 bytes
Saved submission successfully verified.


,index,y
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


,index,y
2495,2495,0
2496,2496,0
2497,2497,0
2498,2498,1
2499,2499,0


## Submission Summary

The submission was generated from the selected equal-weight ensemble:

- CatBoost weight: **0.50**
- XGBoost weight: **0.50**
- OOF decision threshold: **0.130**
- Cross-fitted Balanced Accuracy: **0.8988**
- Predicted test positive rate: **20.36%**

The generated CSV preserves the official sample submission schema and row order.